<a href="https://colab.research.google.com/github/vshalisko/GEE/blob/main/ESE_Plantilla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejemplo para dar una idea soble la plntilla de trabajo Ensayo 2024.12.19


## Ensenada
### Visuzlización de zona de estudio en Colab - GEE
### Vesión con el proceso de generar mosaico para combinar imagenes de varias zonas y/o para eliminar nubes
### Tres fechas de Landsat 8: 2014, 2015, 2020
### Descarga y guardado de Numpy arrays y GeoTIFF de imagen L8 y datos de referencias


In [1]:
## Definir variables

## Prefilo de zona
prefijo = "ESE"

## Ruta en Google Drive (se requiere crear una carpeta en raíz de Google Drive)
ruta = "Colab Data"

## Punto de centro de zona en coordenadas geográficas
punto_interes_text = [-116.6064, 31.8680]

## Proyeccion UTM aplicable
projection_text = "EPSG:32611"

## limites de rectangulo de zona de estudio en UTM
#limites_utm = [531951, 547066, 3513217, 3532728]    # versión agosto 2024
limites_utm = [523989, 548709, 3502560, 3532709]

## tamaño de pixel
pixel = 30

## número de pixeles máximo en segmento
## Nota: GEE generará error en caso que este numero es mayor que 262144
max_tile = 200000

## fechas para imagenes de satelite
#inicio_2015 = "2015-03-01"
#final_2015 = "2015-06-30"

## datos para definición de rango de fechas imagenes de satelite
## rango de meses
m_inicio = 3
m_final = 6
## rangos de años
y_inicio_2014 = 2014
y_final_2014 = 2014
y_inicio_2015 = 2015
y_final_2015 = 2015
y_inicio_2020 = 2020
y_final_2020 = 2020

## porcentaje máximo de nubes permitido
nubes = 4

## bandas de L8 utiles
bands_L8 = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']

## bandas de capa de referencia
bands_B = ['settlement']


#### Praparar el entorno

In [2]:
## cargar bibliotecas GEE y GEEmap
import ee
import geemap
from google.colab import drive
import math
import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal, osr


# Iniciar autentificacion
ee.Authenticate()

# Inicializar proyecto GEE
ee.Initialize(project='ee-vshalisko')

## Inicializar Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Preparar datos para GEE

In [3]:
punto_interes = ee.Geometry.Point(punto_interes_text)

projection = ee.Projection(projection_text)

bbox = ee.Geometry.Rectangle([limites_utm[0], limites_utm[2], limites_utm[1], limites_utm[3]], projection, True, False)

#### Definir funciones auxiliares para descarga de la imagen, control de nubes y calidad de pixeles



In [4]:
## funcion para eliminar nubes y sombras
def maskClouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_bit_mask = (1 << 3)
    cloud_bit_mask = (1 << 4)
    cloud_mask = qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0).And(qa.bitwiseAnd(cloud_bit_mask).eq(0))
    return image.updateMask(cloud_mask)

def descarga_imagen_L8(m_inicio, m_final, y_inicio, y_final, nubes):
  print('\nSelección de la imagen', y_inicio)
  ## Ejemplo de coleccion con presencia de nubes
  L8_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                      .filterBounds(punto_interes)
                      .filter(ee.Filter.calendarRange(m_inicio,m_final, "month"))
                      .filter(ee.Filter.calendarRange(y_inicio, y_final, "year"))
                      .filter(ee.Filter.lessThan("CLOUD_COVER", nubes)))
  print('Capas de datos filtrados', y_inicio, '-', y_final, ':', L8_collection.size().getInfo())

  # Cargar primer imagen de la colección
  L8_image = L8_collection.first()
  image_meta = L8_image.getInfo()
  imagename = image_meta.get('properties',{}).get('LANDSAT_PRODUCT_ID')
  print('Imagen base elegido:', imagename)

  ## obtener lista de nombres de bandas
  bands_original_names = L8_image.bandNames();

  ## Extraer datos de proyeccion de la imagen L8
  projection = L8_image.projection().getInfo();

  ## Consultar datos de la proyección
  print('Proyeccion:', projection.get('crs'))
  print('Transformación lineal:', projection.get('transform'))

  ## generar el mosaico
  L8_image_mosaic = L8_collection.map(maskClouds).reduce(ee.Reducer.mean())

  ## renombrar bandas al esquema original
  L8_image_mosaic = L8_image_mosaic.rename(bands_original_names)

  ## asignar proyeccion al mosaico
  L8_image_re = L8_image_mosaic.reproject(projection.get('crs'), projection.get('transform'), None)

  return L8_image_re

Seleccionar imagenes Landsat 8 para los años 2014, 2015 y 2020

In [5]:
L8_image_2015_re = descarga_imagen_L8(m_inicio, m_final, y_inicio_2015, y_final_2015, nubes)
L8_image_2014_re = descarga_imagen_L8(m_inicio, m_final, y_inicio_2014, y_final_2014, nubes)
L8_image_2020_re = descarga_imagen_L8(m_inicio, m_final, y_inicio_2020, y_final_2020, nubes)


Selección de la imagen 2015
Capas de datos filtrados 2015 - 2015 : 2
Imagen base elegido: LC08_L2SP_039038_20150606_20200909_02_T1
Proyeccion: EPSG:32611
Transformación lineal: [30, 0, 479985, 0, -30, 3628215]

Selección de la imagen 2014
Capas de datos filtrados 2014 - 2014 : 5
Imagen base elegido: LC08_L2SP_039038_20140315_20200912_02_T1
Proyeccion: EPSG:32611
Transformación lineal: [30, 0, 477285, 0, -30, 3628215]

Selección de la imagen 2020
Capas de datos filtrados 2020 - 2020 : 2
Imagen base elegido: LC08_L2SP_039038_20200502_20200820_02_T1
Proyeccion: EPSG:32611
Transformación lineal: [30, 0, 476685, 0, -30, 3628215]


In [ ]:
## Parametros de visualizacion para Landsat 8
vizParams_L8 = {
  'bands': ['SR_B5', 'SR_B4', 'SR_B3'],
   'min': 5000,
   'max': 15000
}

## definir ventana de mapa
map = geemap.Map()

## Centrar
map.centerObject(punto_interes, 10)

map.addLayer(L8_image_2014_re, vizParams_L8, '2014 mosaic FC', False, 0.5)
map.addLayer(punto_interes, {}, "Punto de centro", True, 1)


## Presentar el mapa
map
